# Phase 2A - Silver Tables Ctreation

![image_1783071754982.png](./image_1783071754982.png "image_1783071754982.png")

In [0]:
from pyspark.sql.functions import col, to_date, coalesce, lit
orders    = spark.table("workspace.bronze.orders")
customers = spark.table("workspace.bronze.customers")
order_items  = spark.table("workspace.bronze.order_items")
products     = spark.table("workspace.bronze.products")

### Silver T1 - orders_clean: clean orders + attach the real customer (one row per order)
- **Dedupe** on `order_id` — 0 found, kept as a safe, idempotent habit.
- **Standardise dates**: convert the text timestamp to a real `DATE`.
- Keep only rows where `order_id` and `customer_id` exist — the keys everything joins on.
- Join to customers for **`customer_unique_id`**, the true person behind the order.
- **Grain**: one row per order.

In [0]:
orders_clean = (orders
    .dropDuplicates(["order_id"])                                      # 1. dedupe (safety net)
    .withColumn("order_purchase_date", to_date(col("order_purchase_timestamp")))  # 2. standardise date
    .filter(col("order_id").isNotNull() & col("customer_id").isNotNull())          # 3. keys must exist
    .join(                                                              # 4. bring in the REAL customer id + location
        customers.select("customer_id", "customer_unique_id", "customer_state", "customer_city"),
        on="customer_id", how="left")
    .select("order_id", "customer_id", "customer_unique_id",
            "customer_state", "customer_city", "order_status", "order_purchase_date")
)

orders_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.orders_clean")
print("silver.orders_clean rows:", spark.table("workspace.silver.orders_clean").count())

silver.orders_clean rows: 99441


In [0]:
orders_clean = spark.table("workspace.silver.orders_clean")

### Silver T2 - order_items_enriched: enrich each item with product + order context (one row per item)
- 610 products had no category → relabeled **"unknown"** so their revenue isn't lost.
- **left join** to products (keep the item even if the product is missing); **inner join** to orders (an item needs a date + customer to be useful).
- **Grain**: one row per item — this is our fact table for revenue and lifetime value.

In [0]:
products_clean = (products
    .withColumn("product_category", coalesce(col("product_category_name"), lit("unknown")))  # 610 nulls -> "unknown"
    .select("product_id", "product_category"))

order_items_enriched = (order_items
    .filter(col("order_id").isNotNull() & col("product_id").isNotNull())
    .join(products_clean, on="product_id", how="left")                 # item -> product (keep item if product missing)
    .withColumn("product_category", coalesce(col("product_category"), lit("unknown")))
    .join(orders_clean, on="order_id", how="inner")                    # item -> order (need date + customer)
    .select("order_id", "order_item_id", "customer_unique_id", "customer_state",
            "order_status", "order_purchase_date",
            "product_id", "product_category", "price", "freight_value")
)

order_items_enriched.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.order_items_enriched")
print("silver.order_items_enriched rows:", spark.table("workspace.silver.order_items_enriched").count())

silver.order_items_enriched rows: 112650
